# 01 — English Contextual Embeddings (XLM-RoBERTa)

Generates contextual word embeddings for the **1735 English content words** using
[XLM-RoBERTa-base](https://huggingface.co/xlm-roberta-base) (768 dimensions).

## Strategy

1. **Sentence assignment — sequential text matching (primary)**
   Each content word is matched to one of the 402 sentences by scanning forward
   through the sentence list and checking whether the word (or phrase) appears as
   a consecutive token sequence. A 6-sentence lookahead window handles minor
   ordering discrepancies. Assignment counts prevent over-assigning when the same
   word appears multiple times in a sentence.

2. **Full-scan rescue (secondary)**
   For the ~250 words the sequential pass could not place (capacity exhausted,
   pointer advanced too far, unusual word forms), we scan all 402 sentences for
   any remaining slot and pick the sentence whose onset time is closest to the
   word's timestamp.

3. **Token-level embedding extraction**
   XLM-RoBERTa tokenises each sentence into subword pieces. For each assigned
   content word, we locate the matching token span and average the subword
   vectors to produce one 768-d embedding per word.

**Outputs** (saved to `data/processed/`):
- `en_contextual_aligned_embeddings.csv` — shape (N, 768)
- `en_contextual_matched_indices.csv`    — original word indices for the N kept rows

In [1]:
import pandas as pd
import torch
import numpy as np
import unicodedata
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

print("Loading datasets...")

# ── 1735 content words with timestamps and English text ───────────────────────
word_level_df = pd.read_csv(
    '../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv'
)

# ── 402 English sentences (one per line, CSV with index prefix) ───────────────
with open("../data/sentences/podcast_sentences_en.csv", "r", encoding="utf-8") as f:
    lines = f.readlines()
sentences = [line.strip().split(',', 1)[1] for line in lines[1:] if ',' in line]

# ── 5136-word full transcript with precise word-level timestamps ──────────────
full_transcript = pd.read_csv('../data/ds005574/stimuli/podcast_transcript.csv')

print(f"Content words    : {len(word_level_df)}")
print(f"Sentences        : {len(sentences)}")
print(f"Full transcript  : {len(full_transcript)} words")

/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading datasets...
Content words    : 1735
Sentences        : 402
Full transcript  : 5136 words


## 1. Imports & Data Loading

In [2]:
def normalize(text):
    """
    Normalise a word for text matching:
    - Unicode NFC normalisation
    - Strip curly/straight apostrophes so "didn't" → "didnt",
      "Wikipedia's" → "wikipedias"
    - Strip surrounding punctuation and lowercase
    """
    text = unicodedata.normalize('NFC', str(text))
    text = text.replace('’', '').replace("'", '').replace('`', '')
    return text.strip(' .,!?"()-:;[]{}').lower()


def get_sentence_tokens(sentence, tokenizer, model):
    """
    Tokenise `sentence` with XLM-RoBERTa and return a list of dicts:
      {'text': normalised_word_string, 'vector': 768d numpy array}
    One entry per *word* token (subword pieces averaged within each word).
    """
    encoded = tokenizer(sentence, return_tensors='pt',
                        truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**encoded)

    token_embeddings = outputs.last_hidden_state.squeeze(0)  # (seq_len, 768)
    word_ids   = encoded.word_ids()
    input_ids  = encoded['input_ids'][0]

    word_vectors   = {}
    word_token_ids = {}
    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id not in word_vectors:
            word_vectors[word_id]   = []
            word_token_ids[word_id] = []
        word_vectors[word_id].append(token_embeddings[idx].numpy())
        word_token_ids[word_id].append(input_ids[idx].item())

    tokens = []
    for word_id in sorted(word_vectors.keys()):
        avg_vector = np.mean(word_vectors[word_id], axis=0)
        raw_text   = tokenizer.decode(word_token_ids[word_id])
        norm_text  = normalize(raw_text)
        if norm_text:
            tokens.append({'text': norm_text, 'vector': avg_vector})
    return tokens

## 2. Helper Functions

In [3]:
print("Assigning target words to sentences (sequential text matching)...")

word_to_sentence    = {}
unassignable        = []
assignment_counts   = defaultdict(lambda: defaultdict(int))

sent_idx = 0
for wi in range(len(word_level_df)):
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]
    target_key = '_'.join(components)

    found = False
    for lookahead in range(min(6, len(sentences) - sent_idx)):
        candidate_idx = sent_idx + lookahead

        sent_words = [normalize(w) for w in
                      sentences[candidate_idx].replace('-', ' ').replace('%', ' percent ').split()]

        capacity = 0
        i = 0
        while i <= len(sent_words) - len(components):
            if all(sent_words[i+j] == components[j] for j in range(len(components))):
                capacity += 1; i += len(components)
            else:
                i += 1

        already_assigned = assignment_counts[candidate_idx][target_key]
        if capacity > already_assigned:
            word_to_sentence[wi] = candidate_idx
            assignment_counts[candidate_idx][target_key] += 1
            sent_idx = candidate_idx
            found = True
            break

    if not found:
        unassignable.append(wi)
        word_to_sentence[wi] = None

assigned = sum(1 for v in word_to_sentence.values() if v is not None)
print(f"Assigned by text : {assigned} / {len(word_level_df)}")
print(f"Unassignable     : {len(unassignable)}")

# ── Full-scan rescue for unassigned words ─────────────────────────────────────
# For each unassignable word, scan ALL sentences for remaining capacity and pick
# the one whose timestamp is closest to the word's onset time.
print(f"\nFull-scan rescue for {len(unassignable)} unassigned words...")

# Build a time range for each sentence from already-assigned words
sentence_time_map = {}
for wi, si in word_to_sentence.items():
    if si is None:
        continue
    t_start = word_level_df.iloc[wi]['start']
    t_end   = word_level_df.iloc[wi]['end']
    if si not in sentence_time_map:
        sentence_time_map[si] = [t_start, t_end]
    else:
        sentence_time_map[si][0] = min(sentence_time_map[si][0], t_start)
        sentence_time_map[si][1] = max(sentence_time_map[si][1], t_end)

rescued = 0
still_unassignable = []

for wi in unassignable:
    target_raw = str(word_level_df.iloc[wi]['en']).strip().lower()
    components = [c.strip() for c in target_raw.split('_')]
    target_key = '_'.join(components)
    t = word_level_df.iloc[wi]['start']

    candidates = []
    for si in range(len(sentences)):
        sent_words = [normalize(w) for w in
                      sentences[si].replace('-', ' ').replace('%', ' percent ').split()]
        cap = 0
        i = 0
        while i <= len(sent_words) - len(components):
            if all(sent_words[i+j] == components[j] for j in range(len(components))):
                cap += 1; i += len(components)
            else:
                i += 1

        remaining = cap - assignment_counts[si][target_key]
        if remaining > 0:
            proximity = abs(sentence_time_map[si][0] - t) if si in sentence_time_map else 9999
            candidates.append((proximity, si))

    if candidates:
        candidates.sort()
        best_si = candidates[0][1]
        word_to_sentence[wi] = best_si
        assignment_counts[best_si][target_key] += 1
        rescued += 1
    else:
        still_unassignable.append(wi)

unassignable = still_unassignable
print(f"Rescued  : {rescued}  |  True unassignable: {len(unassignable)}")
print(f"\nTotal assigned : {sum(1 for v in word_to_sentence.values() if v is not None)} / {len(word_level_df)}")

if unassignable:
    print(f"\nWords with no matching sentence anywhere:")
    for wi in unassignable:
        row = word_level_df.iloc[wi]
        print(f"  [{wi:4d}] t={row['start']:.1f}s  '{row['en']}'")

print("\nFirst 10 assignments:")
for wi in range(10):
    si = word_to_sentence[wi]
    print(f"  [{wi:3d}] '{word_level_df.iloc[wi]['en']}'  → sent {si}: '{sentences[si][:65]}'")

Assigning target words to sentences (sequential text matching)...
Assigned by text : 1481 / 1735
Unassignable     : 254

Full-scan rescue for 254 unassigned words...
Rescued  : 225  |  True unassignable: 29

Total assigned : 1706 / 1735

Words with no matching sentence anywhere:
  [  50] t=54.0s  'mean'
  [ 165] t=151.2s  'trying'
  [ 291] t=270.0s  'last_years'
  [ 329] t=307.1s  'wanting'
  [ 418] t=397.7s  'fair_use'
  [ 427] t=409.1s  'alls'
  [ 677] t=643.6s  'cause'
  [ 807] t=761.4s  'created'
  [ 894] t=858.1s  'need'
  [ 934] t=910.0s  'cause'
  [ 953] t=931.3s  'know'
  [1039] t=1021.8s  'beaching'
  [1089] t=1073.7s  'dash'
  [1095] t=1081.6s  'afternoon'
  [1098] t=1083.2s  'federal_court'
  [1100] t=1085.6s  'unanticipated'
  [1140] t=1128.1s  'answered'
  [1177] t=1163.7s  'fourteenth'
  [1195] t=1181.2s  'endgame'
  [1298] t=1306.8s  'know'
  [1404] t=1424.8s  'cause'
  [1408] t=1430.6s  'part'
  [1447] t=1481.6s  'appeals_court'
  [1462] t=1494.5s  'honor'
  [1470] t=15

## 3. Assign Content Words to Sentences

In [4]:
print("Loading XLM-RoBERTa-base...")
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model     = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()
print("Model ready.")

Loading XLM-RoBERTa-base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]Error processing line 1 of /Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "/Users/YAHLIZ/miniforge3/envs/language_project_env/lib/python3.10/site.py", line 195, in addpackage
      exec(line)
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8461.57it/s]
XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

Model ready.


## 4. Load XLM-RoBERTa

In [5]:
print("Extracting embeddings...\n")

final_aligned_embeddings = []
matched_word_indices     = []
dropped_word_indices     = []

# ── Cache sentence tokenisations (each sentence only processed once) ──────────
sentence_token_cache = {}

def get_tokens_cached(si):
    if si not in sentence_token_cache:
        sentence_token_cache[si] = get_sentence_tokens(sentences[si], tokenizer, model)
    return sentence_token_cache[si]

# ── Group assigned words by sentence; words with si=None are immediately dropped
words_per_sentence = defaultdict(list)
for wi, si in word_to_sentence.items():
    if si is not None:
        words_per_sentence[si].append(wi)
    else:
        dropped_word_indices.append(wi)

# ── Per-sentence extraction ───────────────────────────────────────────────────
assignment_counts = defaultdict(lambda: defaultdict(int))

for si in sorted(words_per_sentence.keys()):
    target_word_indices = sorted(words_per_sentence[si])
    sentence_tokens     = get_tokens_cached(si)
    n_tokens            = len(sentence_tokens)
    consumed_positions  = set()   # token positions already claimed by a word
    sentence_pointer    = 0       # forward scan position

    for word_idx in target_word_indices:
        target_raw        = str(word_level_df.iloc[word_idx]['en']).strip().lower()
        target_components = [c.strip() for c in target_raw.split('_')]
        phrase_len        = len(target_components)

        def try_match_at(start):
            """Return True if the phrase matches at token position `start`."""
            if start + phrase_len > n_tokens:
                return False
            if set(range(start, start + phrase_len)) & consumed_positions:
                return False
            return all(
                sentence_tokens[start + i]['text'] == target_components[i]
                for i in range(phrase_len)
            )

        # Pass 1: scan forward from the current pointer
        matched_at = None
        for start in range(sentence_pointer, n_tokens):
            if try_match_at(start):
                matched_at = start
                break

        # Pass 2: scan backward (catches repeated words already passed)
        if matched_at is None:
            for start in range(0, sentence_pointer):
                if try_match_at(start):
                    matched_at = start
                    break

        if matched_at is not None:
            phrase_vectors = [sentence_tokens[matched_at + i]['vector']
                              for i in range(phrase_len)]
            final_aligned_embeddings.append(np.mean(phrase_vectors, axis=0))
            matched_word_indices.append(word_idx)
            for i in range(phrase_len):
                consumed_positions.add(matched_at + i)
            if matched_at >= sentence_pointer:
                sentence_pointer = matched_at + phrase_len
        else:
            dropped_word_indices.append(word_idx)

Extracting embeddings...



## 5. Extraction Loop

In [6]:
print("=" * 55)
print("ENGLISH ALIGNMENT REPORT")
print("=" * 55)

extracted_count       = len(final_aligned_embeddings)
dropped_in_extraction = len(dropped_word_indices)

print(f"Total target words       : {len(word_level_df)}")
print(f"Successfully matched     : {extracted_count}")
print(f"Dropped                  : {dropped_in_extraction}")
print(f"Total accounted for      : {extracted_count + dropped_in_extraction}")
print(f"Match rate (of total)    : {extracted_count / len(word_level_df) * 100:.1f}%")

if dropped_word_indices:
    print(f"\nDropped words:")
    for idx in dropped_word_indices:
        row = word_level_df.iloc[idx]
        si  = word_to_sentence.get(idx)
        sent_preview = sentences[si][:60] if si is not None else 'NONE'
        print(f"  [{idx:4d}] '{row['en']}'  -> sent {si}: '{sent_preview}'")

ENGLISH ALIGNMENT REPORT
Total target words       : 1735
Successfully matched     : 1692
Dropped                  : 43
Total accounted for      : 1735
Match rate (of total)    : 97.5%

Dropped words:
  [  50] 'mean'  -> sent None: 'NONE'
  [ 165] 'trying'  -> sent None: 'NONE'
  [ 291] 'last_years'  -> sent None: 'NONE'
  [ 329] 'wanting'  -> sent None: 'NONE'
  [ 418] 'fair_use'  -> sent None: 'NONE'
  [ 427] 'alls'  -> sent None: 'NONE'
  [ 677] 'cause'  -> sent None: 'NONE'
  [ 807] 'created'  -> sent None: 'NONE'
  [ 894] 'need'  -> sent None: 'NONE'
  [ 934] 'cause'  -> sent None: 'NONE'
  [ 953] 'know'  -> sent None: 'NONE'
  [1039] 'beaching'  -> sent None: 'NONE'
  [1089] 'dash'  -> sent None: 'NONE'
  [1095] 'afternoon'  -> sent None: 'NONE'
  [1098] 'federal_court'  -> sent None: 'NONE'
  [1100] 'unanticipated'  -> sent None: 'NONE'
  [1140] 'answered'  -> sent None: 'NONE'
  [1177] 'fourteenth'  -> sent None: 'NONE'
  [1195] 'endgame'  -> sent None: 'NONE'
  [1298] 'know'  -

## 7. Save

In [7]:
if len(final_aligned_embeddings) > 0:
    out_emb = '../data/processed/en_contextual_aligned_embeddings.csv'
    out_idx = '../data/processed/en_contextual_matched_indices.csv'

    # ── Save embeddings matrix ────────────────────────────────────────────────
    embeddings_df = pd.DataFrame(final_aligned_embeddings)
    embeddings_df.to_csv(out_emb, index=False)

    # ── Save index map (original word positions kept) ─────────────────────────
    pd.DataFrame({'original_word_idx': matched_word_indices}).to_csv(out_idx, index=False)

    print(f"Saved embeddings  -> {out_emb}  shape: {embeddings_df.shape}")
    print(f"Saved index map   -> {out_idx}")

    # ── Verification read-back ────────────────────────────────────────────────
    emb = pd.read_csv(out_emb)
    idx = pd.read_csv(out_idx)
    print(f"\nEmbeddings shape  : {emb.shape}")
    print(f"Index map shape   : {idx.shape}")
    print(f"Index range       : {idx['original_word_idx'].min()} – {idx['original_word_idx'].max()}")
    print(f"Any NaN           : {emb.isnull().any().any()}")
    print(f"\nFirst 5 indices   : {idx['original_word_idx'].tolist()[:5]}")
    print(f"Sample row 0 (first 5 dims): {emb.iloc[0, :5].tolist()}")

Saved embeddings  -> ../data/processed/en_contextual_aligned_embeddings.csv  shape: (1692, 768)
Saved index map   -> ../data/processed/en_contextual_matched_indices.csv

Embeddings shape  : (1692, 768)
Index map shape   : (1692, 1)
Index range       : 0 – 1734
Any NaN           : False

First 5 indices   : [0, 1, 2, 3, 4]
Sample row 0 (first 5 dims): [0.042788513, 0.047795862, 0.008115685, 0.005766019, 0.022064418]
